# Video Clone Douyin - Google Colab GPU Backend

Notebook này khởi chạy backend FastAPI của **Video Clone Douyin** trên GPU Google Colab và xuất URL public qua Cloudflare Tunnel.

Cách dùng:
1. Chọn **Runtime > Change runtime type > GPU**.
2. Chạy lần lượt các cell bên dưới.
3. Sao chép URL `https://...trycloudflare.com`.
4. Mở app desktop, vào **Cài đặt > Google Colab GPU**, dán URL và bấm **Kiểm tra** rồi chọn **Remote Google Colab**.


In [ ]:
#@title 1. Cấu hình repo và kiểm tra GPU
import os, subprocess, textwrap

# Repo Colab runtime riêng, chỉ chứa backend cần thiết để tránh đưa toàn bộ code desktop lên Colab.
REPO_URL = "https://github.com/nqthaivl/videocolab.git"  #@param {type:"string"}
PROJECT_DIR = "/content/videocolab"
PORT = 3900

print("GPU Colab:")
!nvidia-smi || true

if not os.path.isdir(PROJECT_DIR):
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print("Repo đã tồn tại, cập nhật code mới nhất...")
    %cd "$PROJECT_DIR"
    !git pull --ff-only || true

%cd "$PROJECT_DIR"
print("Thư mục dự án:", os.getcwd())


In [ ]:
#@title 2. Cài dependencies backend
import os

print("Cài system packages...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg git wget

print("Cài uv và Python dependencies từ pyproject.toml...")
!python -m pip install -U pip uv
!uv pip install --system -e .

print("Tải cloudflared...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("Hoàn tất cài đặt.")


In [ ]:
#@title 3. Khởi chạy backend và Cloudflare Tunnel
import os, re, subprocess, time, requests, signal

DATA_DIR = "/content/video-clone-data"
CACHE_DIR = "/content/video-clone-models"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "OMNIVOICE_DATA_DIR": DATA_DIR,
    "OMNIVOICE_CACHE_DIR": CACHE_DIR,
    "HF_HOME": CACHE_DIR,
    "HF_HUB_CACHE": CACHE_DIR,
    "TORCH_HOME": CACHE_DIR,
    "OMNIVOICE_MCP_DISABLE": "1",
    "OMNIVOICE_UI_PORT": "5174",
    "OMNIVOICE_ALLOWED_ORIGINS": "http://127.0.0.1:5174,http://localhost:5174,null",
    "VIDEO_DUBBING_DISABLE_MODEL_PRELOAD": "1",
    "OMNIVOICE_ACTIVATED": "1",
    "HF_HUB_DISABLE_XET": "1",
}

backend_cmd = ["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT)]
print("Khởi chạy backend Video Clone...")
backend = subprocess.Popen(backend_cmd, cwd=os.path.join(PROJECT_DIR, "backend"), env=env)

health_url = f"http://127.0.0.1:{PORT}/health"
for i in range(120):
    try:
        r = requests.get(health_url, timeout=2)
        if r.ok:
            print("Backend sẵn sàng:", r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Backend không phản hồi /health sau 120 giây")

print("Khởi tạo Cloudflare Tunnel...")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", health_url.rsplit("/", 1)[0]],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

try:
    public_url = None
    while True:
        line = tunnel.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        if "trycloudflare.com" in line or "error" in line.lower():
            print("[cloudflared]", line.strip())
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and public_url is None:
            public_url = match.group(0)
            print("\n" + "=" * 72)
            print("VIDEO CLONE COLAB BACKEND ĐÃ SẴN SÀNG")
            print("URL API Colab:")
            print(public_url)
            print("\nDán URL này vào app: Cài đặt > Google Colab GPU > URL API Colab")
            print("=" * 72 + "\n")
except KeyboardInterrupt:
    print("Dừng backend và tunnel...")
finally:
    tunnel.terminate()
    backend.terminate()
